# Index-selection Knowledge Indicators with the Context Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kderusso/elasticsearch-labs/blob/kderusso/context-engine-notebooks/notebooks/context-engine/manual-walkthrough/index-metadata-kis.ipynb)

The **Context Engine** gives AI agents structured, pre-computed **Knowledge Indicators (KIs)**, facts they can retrieve in a single call instead of scanning thousands of documents. This interactive notebook will walk you through how to set up a context engine, using the official [Elasticsearch Python client](https://www.elastic.co/guide/en/elasticsearch/client/python-api/current/index.html) and the [Kibana Workflows API](https://www.elastic.co/guide/en/kibana/current/index.html).

For this walkthrough, we index three [BEIR benchmark](https://github.com/beir-cellar/beir) corpora covering distinct domains, profile each into an index-level KI, and show how routing with the Context Engine replaces scatter-gather search.


## Create Elasticsearch environment

This notebook is designed to be run on an Elastic Serverless project. This functionality will be available in a future Elastic cloud release. If you don't have an Elastic Serverless project, sign up [here](https://cloud.elastic.co/registration?onboarding_token=search&cta=cloud-registration&tech=trial&plcmt=article%20content&pg=search-labs).

Once logged in to your Elastic Cloud account, go to the Create page and select Create project.

## Install packages and import modules

To get started, we'll need to connect to our Elastic deployment using the Python client. We connect using the Elasticsearch and Kibana endpoint URLs and an API key, which works for both Cloud deployments and Serverless projects.

In [ ]:
!pip install -q "elasticsearch>=9,<10" datasets requests pyyaml ipywidgets langchain-openai langgraph deepagents

### Initialize the Elasticsearch client

Instantiate the Elasticsearch Python client with your **Elasticsearch** and **Kibana** endpoint URLs and an API key — the same inputs work for both an Elastic Cloud deployment and a Serverless project (both expose direct endpoint URLs on their deployment/project page).

- [Create an API key](https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key)

In [ ]:
import json
import time
import requests
from getpass import getpass
from elasticsearch import Elasticsearch, helpers

ES_URL = input("Elasticsearch endpoint URL: ").strip().rstrip("/")
KIBANA_URL = input("Kibana endpoint URL: ").strip().rstrip("/")
ELASTIC_API_KEY = getpass("Elastic API key: ")

client = Elasticsearch(hosts=[ES_URL], api_key=ELASTIC_API_KEY)
print(client.info())

Next, we ensure that Kibana is accessible.

In [ ]:
def kbn_request(method, path, *, body=None, internal=False, api_version=None):
    """Call a Kibana REST API and return the parsed JSON response."""
    headers = {
        "Authorization": f"ApiKey {ELASTIC_API_KEY}",
        "kbn-xsrf": "true",
        "Content-Type": "application/json",
    }
    if internal:
        headers["x-elastic-internal-origin"] = "Kibana"
    if api_version:
        headers["elastic-api-version"] = api_version
    resp = requests.request(
        method,
        f"{KIBANA_URL}{path}",
        headers=headers,
        data=json.dumps(body) if body is not None else None,
    )
    resp.raise_for_status()
    return resp.json() if resp.text else {}

status = kbn_request("GET", "/api/status")
print("Kibana status:", status.get("status", {}).get("overall", {}).get("level"))

### Enable the Context Engine feature flag

The Context Engine is gated behind the `contextEngine:enabled` advanced setting. We will enable this flag via the settings API.

> If the cell prints "Could not set feature flag", you may not be able to set this feature flag via the API. Verify that the `contextEngine:enabled` feature flag is toggled ON in Stack Management → Advanced Settings.

In [ ]:
try:
    kbn_request("POST", "/internal/kibana/settings",
                body={"changes": {"contextEngine:enabled": True}}, internal=True)
    print("Enabled feature flag: contextEngine:enabled")
except Exception as e:
    print(f"Could not set feature flag via API: {e}")
    print("Verify that the `contextEngine:enabled` feature flag is toggled ON in Stack Management → Advanced Settings.")

### Set up the deep agent harness

Both answers in this notebook come from the same [LangChain deep agents](https://pypi.org/project/deepagents/) harness: same model, same system prompt, same question. The only thing we swap between the baseline and the Context Engine is the search tool the agent gets. Each `ask_agent` call also prints the tokens the agent spent (summed over every model call in the run), so you can compare the inference cost of the two paths. Note the KI path has an indexing-time cost this doesn't capture: the workflow's `ai.agent` step already spent tokens generating each KI. The harness talks to any OpenAI-compatible endpoint — press Enter at the prompts to accept the defaults (OpenRouter + Claude Haiku), or point it at your own provider and model.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.callbacks import get_usage_metadata_callback
from deepagents import create_deep_agent

# Any OpenAI-compatible endpoint works (OpenRouter, OpenAI, a local vLLM/Ollama, ...).
# Press Enter at the prompts to accept the defaults: OpenRouter + Claude Haiku.
LLM_BASE_URL = input("LLM base URL [https://openrouter.ai/api/v1]: ").strip() or "https://openrouter.ai/api/v1"
LLM_MODEL = input("LLM model [anthropic/claude-haiku-4-5]: ").strip() or "anthropic/claude-haiku-4-5"
LLM_API_KEY = getpass("LLM API key: ")

# The deep agent plans and may chain several tool calls, so give it output headroom.
agent_llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
    max_tokens=4096,
)

SYSTEM_PROMPT = """You are a research assistant answering questions about several document
collections. You have NOT memorized them. Whenever a question depends on specific facts, call
your search tools to retrieve relevant material before answering. Ground your answer strictly
in what the tools return, and cite the titles of the sources you used. If the retrieved context
does not contain the answer, say so plainly rather than guessing."""

def ask_agent(question, search_tools):
    """Build a deep agent around the given search tools and ask it a question."""
    agent = create_deep_agent(model=agent_llm, tools=search_tools, system_prompt=SYSTEM_PROMPT)
    with get_usage_metadata_callback() as cb:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    usage = next(iter(cb.usage_metadata.values()), {})
    print(f"[tokens] input={usage.get('input_tokens', 0):,}  "
          f"output={usage.get('output_tokens', 0):,}  total={usage.get('total_tokens', 0):,}\n")
    return result["messages"][-1].content

## Index sample BEIR datasets (BM25 only)

We use three [BEIR benchmark](https://github.com/beir-cellar/beir) corpora from different domains. We stream 100 documents per dataset from Hugging Face.

| Dataset | Domain | Index name |
|---------|--------|------------|
| [FiQA](https://huggingface.co/datasets/BeIR/fiqa) | Financial Q&A | `beir-fiqa` |
| [NFCorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | Biomedical / nutrition | `beir-nfcorpus` |
| [SciFact](https://huggingface.co/datasets/BeIR/scifact) | Scientific fact-checking | `beir-scifact` |

We generate titles if they do not already exist in the corpora documents, and attach `_meta.description` and per-field `meta.description` to each index — this human-written context is what the profiling workflow reads.

In [ ]:
from datasets import load_dataset

SAMPLE_DOCS = 100

DATASETS = [
    {
        "hf_dataset": "BeIR/fiqa",
        "hf_config": "corpus",
        "hf_split": "corpus",
        "index_name": "beir-fiqa",
        "meta_description": (
            "FiQA: financial question answering corpus from StackExchange Finance "
            "community posts and web crawls. Covers investments, banking, taxes, "
            "and market analysis. BM25-only index."
        ),
    },
    {
        "hf_dataset": "BeIR/nfcorpus",
        "hf_config": "corpus",
        "hf_split": "corpus",
        "index_name": "beir-nfcorpus",
        "meta_description": (
            "NFCorpus: biomedical information retrieval corpus from NutritionFacts.org. "
            "Contains nutrition science and medical research documents on diet, disease, "
            "and health interventions. BM25-only index."
        ),
    },
    {
        "hf_dataset": "BeIR/scifact",
        "hf_config": "corpus",
        "hf_split": "corpus",
        "index_name": "beir-scifact",
        "meta_description": (
            "SciFact: scientific fact-checking corpus of biomedical research abstracts "
            "used to verify factual claims in peer-reviewed literature. BM25-only index."
        ),
    },
]

for ds in DATASETS:
    idx = ds["index_name"]
    client.indices.delete(index=idx, ignore_unavailable=True)
    client.indices.create(
        index=idx,
        mappings={
            "_meta": {"description": ds["meta_description"]},
            "properties": {
                "title": {"type": "text", "meta": {"description": "Document or article title."}},
                "text": {"type": "text", "meta": {"description": "Full document body text."}},
            },
        },
    )

    corpus = load_dataset(
        ds["hf_dataset"], ds["hf_config"], split=ds["hf_split"], streaming=True
    )

    def actions(corpus, index_name, n):
        for i, row in enumerate(corpus):
            if i >= n:
                break
            yield {
                "_index": index_name,
                "_id": row["_id"],
                "_source": {"title": row.get("title", "").strip() or " ".join(row.get("text", "").split())[:100], "text": row.get("text", "")},
            }

    helpers.bulk(client, actions(corpus, idx, SAMPLE_DOCS))
    client.indices.refresh(index=idx)
    print(f"Indexed {client.count(index=idx)['count']} documents into '{idx}'.")


## Ask a question with the deep agent

Without index-level KIs there is no way to know which of the three indices is relevant for a given question. The agent's only tool searches all of them and returns the merged results.

In [ ]:
QUESTION = "What investment strategies minimize long-term financial risk?"

@tool
def search_all_indices(query: str) -> str:
    """Search every available document index with BM25 (lexical match over title and text).

    Call this whenever answering depends on facts from the document collections. There is
    no way to tell which index holds the answer, so this searches all of them and returns
    the merged top documents, each labeled with the index it came from.
    """
    chunks = []
    for ds in DATASETS:
        idx = ds["index_name"]
        resp = client.search(
            index=idx,
            body={
                "query": {"multi_match": {"query": query, "fields": ["title", "text"]}},
                "size": 3,
                "_source": ["title", "text"],
            },
        )
        for hit in resp["hits"]["hits"]:
            src = hit["_source"]
            chunks.append(f"[from {idx}] {src.get('title', '')}\n{src.get('text', '')[:500]}")
    if not chunks:
        return "No matching documents found."
    return "\n\n".join(chunks)

print("=== Agent answer: scatter-gather across all indices ===\n")
print(ask_agent(QUESTION, [search_all_indices]))

## Create a workflow to generate Knowledge Indicators

This workflow profiles each index and writes a routing KI into the Context Engine:

| Step | Type | What it does |
|------|------|--------------|
| `get_mapping` | `elasticsearch.request` | Read the mapping, including `_meta.description` and per-field descriptions. |
| `sample_docs` | `elasticsearch.search` | Pull a few real documents so the profile reflects actual value shapes. |
| `profile_index` | `ai.agent` | Generate a structured index profile as structured output. |
| `sink_index_ki` | `contextEngine.addEntry` | Write the profile into the Context Engine as a KI. |

### GenAI connector (optional)

Leave blank to use Kibana's default GenAI connector, or paste a connector id to pin one.

In [ ]:
LLM_CONNECTOR_ID = input("Kibana GenAI connector id (blank = default connector): ").strip()

In [ ]:
_WORKFLOW_YAML_TEMPLATE = """
version: '1'
name: beir-index-profile-ki
description: Profile an index into an index-selection Knowledge Indicator.
enabled: true
tags:
  - context-engine
  - index-selection
triggers:
  - type: manual
    inputs:
      - name: indexName
        type: string
        required: true
        description: The Elasticsearch index to profile.
steps:
  - name: get_mapping
    type: elasticsearch.request
    with:
      method: GET
      path: '/{{ inputs.indexName }}/_mapping'

  - name: sample_docs
    type: elasticsearch.search
    with:
      index: '{{ inputs.indexName }}'
      size: 3
      query:
        match_all: {}

  - name: profile_index
    type: ai.agent
    connector-id: "__LLM_CONNECTOR_ID__"
    timeout: 120s
    with:
      message: >
        You are a data steward building an INDEX PROFILE for an enterprise data
        catalog. Downstream, an AI agent uses these profiles to decide WHICH
        Elasticsearch index to query for a given user question - this is an
        index-SELECTION aid, not a place to answer the question itself.

        Ground everything in the provided mapping + samples. Never invent fields,
        values, or purpose. If unknown, use an empty string/array. Optimize for
        routing: make it obvious what questions this index can authoritatively
        answer, and what it cannot. Prefer concrete field names and real example
        values over vague phrasing.

        Index name: {{ inputs.indexName }}

        Elasticsearch mapping (JSON):
        {{ steps.get_mapping.output | json }}

        Sample documents (JSON):
        {{ steps.sample_docs.output.hits.hits | map: '_source' | json }}
      schema:
        type: object
        properties:
          display_name:
            type: string
            description: A concise human-readable name for what this index represents (<= 8 words).
          purpose:
            type: string
            description: 2-4 sentences describing what this index stores and its role. PRIMARY semantic surface for matching a question to this index.
          answers_questions:
            type: array
            items:
              type: string
            description: 3-7 representative natural-language questions this index can authoritatively answer.
          does_not_contain:
            type: array
            items:
              type: string
            description: 1-4 things a searcher might wrongly expect here but that live elsewhere, to prevent mis-routing.
          key_fields:
            type: array
            items:
              type: string
            description: 3-10 of the most query-relevant fields as "field_name - what it is".
          when_to_use:
            type: string
            description: A single crisp routing heuristic - when should an agent pick THIS index? (<= 30 words).
          example_esql:
            type: string
            description: One realistic, runnable ES|QL query against this index answering one of answers_questions.
        required:
          - display_name
          - purpose
          - answers_questions
          - key_fields
          - when_to_use

  - name: sink_index_ki
    type: contextEngine.addEntry
    with:
      originId: '{{ inputs.indexName }}'
      attachmentType: corpus_entry
      action: upsert
      chunks:
        - type: corpus_entry
          title: '{{ steps.profile_index.output.structured_output.display_name | default: inputs.indexName }}'
          content: >
            === SOURCE / PROVENANCE ===
            This is an INDEX PROFILE for routing/index-selection.
            Backing Elasticsearch index: {{ inputs.indexName }}
            Inspect it directly with ES|QL:
            FROM {{ inputs.indexName }} | LIMIT 10
            === WHAT THIS INDEX IS ===
            {{ steps.profile_index.output.structured_output.purpose }}
            Questions this index can answer: {{ steps.profile_index.output.structured_output.answers_questions | join: " | " }}
            When to use this index: {{ steps.profile_index.output.structured_output.when_to_use }}
            Example query:
            {{ steps.profile_index.output.structured_output.example_esql }}
          description: >
            Index profile: {{ steps.profile_index.output.structured_output.display_name }}.
            Does NOT contain: {{ steps.profile_index.output.structured_output.does_not_contain | join: "; " }}.
            Key fields: {{ steps.profile_index.output.structured_output.key_fields | join: "; " }}.

  - name: output_result
    type: workflow.output
    with:
      indexName: '{{ inputs.indexName }}'
      display_name: '{{ steps.profile_index.output.structured_output.display_name }}'
      when_to_use: '{{ steps.profile_index.output.structured_output.when_to_use }}'
"""

# Pin a connector if one was supplied.
if LLM_CONNECTOR_ID:
    WORKFLOW_YAML = _WORKFLOW_YAML_TEMPLATE.replace("__LLM_CONNECTOR_ID__", LLM_CONNECTOR_ID)
else:
    WORKFLOW_YAML = "\n".join(
        line for line in _WORKFLOW_YAML_TEMPLATE.splitlines() if "__LLM_CONNECTOR_ID__" not in line
    )

print(WORKFLOW_YAML)

In [ ]:
WF_API_VERSION = "2023-10-31"
TERMINAL_STATES = {"completed", "failed", "cancelled", "timed_out", "skipped"}

def create_workflow(yaml_str):
    return kbn_request("POST", "/api/workflows/workflow",
                       body={"yaml": yaml_str}, api_version=WF_API_VERSION)

def run_workflow(workflow_id, inputs):
    return kbn_request("POST", f"/api/workflows/workflow/{workflow_id}/run",
                       body={"inputs": inputs}, api_version=WF_API_VERSION)

def wait_for_execution(execution_id, timeout=300, interval=3):
    deadline = time.time() + timeout
    while time.time() < deadline:
        ex = kbn_request("GET", f"/api/workflows/executions/{execution_id}?includeOutput=true",
                         api_version=WF_API_VERSION)
        if ex["status"] in TERMINAL_STATES:
            return ex
        time.sleep(interval)
    raise TimeoutError(f"Execution {execution_id} did not finish within {timeout}s")

wf = create_workflow(WORKFLOW_YAML)
workflow_id = wf["id"]
print("Created workflow:", workflow_id)

This workflow will generate and index Knowledge Indicators. This step may take a few minutes to run.

In [ ]:
execution_results = {}

for ds in DATASETS:
    idx = ds["index_name"]
    run_resp = run_workflow(workflow_id, {"indexName": idx})
    exec_id = run_resp["workflowExecutionId"]
    print(f"Profiling '{idx}' — execution: {exec_id}")

    result = wait_for_execution(exec_id)
    execution_results[idx] = result
    print(f"  Status: {result['status']}")

    if result["status"] == "completed":
        output = result.get("context", {}).get("output", {})
        print(f"  Profile: {output.get('display_name', '—')} — {output.get('when_to_use', '—')}")
    else:
        if result.get("error"):
            print("  Error:", result["error"].get("message", result["error"]))
        for step in result.get("stepExecutions", []):
            if step.get("status") == "failed":
                print("  Failed step:", step.get("stepId"), "->", json.dumps(step.get("error")))

## Retrieve Knowledge Indicators with `get_context`

With index-level KIs in the Context Engine, one `get_context` call identifies the right index. First, a thin wrapper over the Context Engine's `_search` endpoint — each result is an index-profile KI written by the workflow.

TODO: Replace with a call to `get_context` after the API is renamed. 

In [ ]:
def get_context(query, size=5, types=None):
    body = {
        "query": query,
        "size": size,
        "fields": ["content", "description", "tags", "references"],
    }
    if types:
        body["filters"] = {"types": types}
    return kbn_request("POST", "/internal/agent_context_layer/sml/_search",
                       body=body, internal=True)

## Ask the same question — the agent routes with KIs

Now the Context Engine side of the comparison: same agent harness, same question, different tools. `search_context` searches the index-profile KIs so the agent can decide which index holds the answer; `search_index` then runs a targeted search against only that index.

In [ ]:
@tool
def search_context(query: str) -> str:
    """Search the Context Engine for index-profile Knowledge Indicators.

    Call this FIRST to decide which Elasticsearch index is relevant to the question.
    Pass a focused natural-language query. Returns the most relevant index profiles with
    their title and content (each names the backing index it describes).
    """
    results = get_context(query, size=5, types=["corpus_entry"])["results"]
    if not results:
        return "No relevant context found in the Context Engine."
    return "\n\n".join(f"[{r['title']}]\n{r.get('content', '')}" for r in results)

@tool
def search_index(index_name: str, query: str) -> str:
    """Run a BM25 search against one specific Elasticsearch index.

    Call this AFTER search_context has identified the relevant index, passing that
    index's name. Returns the top matching documents as title + text excerpt.
    """
    resp = client.search(
        index=index_name,
        body={
            "query": {"multi_match": {"query": query, "fields": ["title", "text"]}},
            "size": 3,
            "_source": ["title", "text"],
        },
    )
    hits = resp["hits"]["hits"]
    if not hits:
        return "No matching documents found."
    return "\n\n".join(
        f"[{h['_source'].get('title', '')}]\n{h['_source'].get('text', '')[:500]}"
        for h in hits
    )

print("=== Agent answer: route via KIs, then search one index ===\n")
print(ask_agent(QUESTION, [search_context, search_index]))

## Clean up

Remove the KIs from the Context Engine, delete the workflow, and drop the Elasticsearch index. The feature flag remains enabled unless you explicitly disable it.

In [ ]:
index_names = {ds["index_name"] for ds in DATASETS}
for hit in get_context("index profile", size=50, types=["corpus_entry"])["results"]:
    ki_type, _, origin_id = hit.get("origin", {}).get("uri", "").partition("://")
    if origin_id in index_names:
        kbn_request("DELETE", f"/internal/agent_context_layer/sml/{ki_type}/{origin_id}", internal=True)
        print("Deleted KI:", f"{ki_type}://{origin_id}", "—", hit.get("title"))

try:
    kbn_request("DELETE", f"/api/workflows/workflow/{workflow_id}", api_version=WF_API_VERSION)
    print("Deleted workflow:", workflow_id)
except Exception as e:
    print("Workflow delete skipped:", e)

for ds in DATASETS:
    client.indices.delete(index=ds["index_name"], ignore_unavailable=True)
    print("Deleted index:", ds["index_name"])